# GLM-HMM finetuned model visualization

Per model type and subject group, using the finetuned models from `5.30`
(`<model_dir>/<group>_final.pkl`):

- **GLM weights** — per-session refits faint; the dark line is the **pooled** model for pooled-CV
  models (clean group estimate) or the across-session mean ± std for session-CV models. Weights are
  plotted as `-W` (the direction toward choice = 1), since per-session multi-state fits are
  data-starved and noisy.
- **Psychometric** — actual vs model `P(choice = 1)` against the stimulus, pooled over the group's
  sessions (model = posterior-weighted choice probability).
- **Mean transition matrix** — averaged across the group's session models.
- **State occupancy** — fractional posterior occupancy per state, each session faint, mean dark.

States are matched by index across sessions; this is well-justified for pooled-CV models (every
session shares the pooled-fold init) and approximate for session-CV models.

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit

from imports import *
from config import dir_config, main_config

In [ ]:
processed_dir = Path(dir_config.data.processed)
glm_hmm_dir = processed_dir / "glm_hmm"

MODEL_PATHS = {f.stem: f for f in glm_hmm_dir.iterdir() if f.is_dir()}


def model_label(model_name):
    # The model dir is named by its experiment key (alias) + CV suffix; strip the suffix.
    return model_name.replace("__global_pooled_cv", "").replace("__session_pooled_cv", "")


desired_order = ["asmHC", "Tremor_OFF", "Brady_OFF", "Tremor_ON", "Brady_ON"]
order_map = {name: i for i, name in enumerate(desired_order)}

STATE_COLORS = ["#ff7f00", "#4daf4a", "#4dafda", "#373e48", "#984ea3", "#e41a1c"]

In [ ]:
def load_finals(model_path):
    """{group: bundle} for every <group>_final.pkl saved by 5.30."""
    out = {}
    for p in sorted(model_path.glob("*_final.pkl")):
        out[p.stem[: -len("_final")]] = pickle.load(open(p, "rb"))
    return out


def state_colors(K):
    if K <= len(STATE_COLORS):
        return STATE_COLORS[:K]
    return [plt.cm.tab20(i % 20) for i in range(K)]


def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def session_design(bundle, session_id):
    """(inputs, choices, mask, valid, stimulus) for one session, from the saved data frame."""
    df = bundle["data"][session_id]
    feats = bundle["config"]["model_features"]
    inputs = np.asarray(df[feats], dtype=float)
    choices = df["choices"].values.reshape(-1, 1).astype(int)
    mask = df["mask"].values.reshape(-1, 1).astype(bool)
    valid = ~df["invalid_idx"].values
    stimulus = df["stimulus"].values
    return inputs, choices, mask, valid, stimulus


def choice_prob(model, inputs, choices, mask):
    """Posterior-weighted marginal P(choice = 1 | x_t) under the fitted GLM-HMM.

    P(y=1 | state k, x) = sigmoid(-x . W_k), matching ssm's input_driven_obs convention
    (hence the weight plots use -W); states are marginalised with the posterior P(state).
    """
    neg_w = -np.array(model.observations.params)[:, 0, :]            # (K, D)
    p1_state = expit(inputs @ neg_w.T)                               # (T, K)
    Ez = model.expected_states(choices, input=inputs, mask=mask)[0]  # (T, K)
    return (Ez * p1_state).sum(1)                                    # (T,)

In [ ]:
def plot_weights(ax, bundle):
    """Per-session GLM weights (faint) with the group-level estimate dark.

    For pooled-CV models the dark line is the single pooled model's weights (clean group estimate);
    for session-CV models it is the across-session mean +/- std.
    """
    K, feats = bundle["best_k"], bundle["config"]["model_features"]
    cols = state_colors(K)
    weights = []
    for model in bundle["model"]["models"].values():
        w = -np.array(model.observations.params).reshape(K, -1)   # (K, D), toward choice = 1
        weights.append(w)
        for k in range(K):
            ax.plot(range(len(feats)), w[k], marker="o", color=cols[k], ls="-", ms=3, lw=1, alpha=0.15)
    weights = np.array(weights)

    pooled = bundle["model"].get("pooled")
    if pooled is not None:
        Wp = -np.array(pooled.observations.params).reshape(K, -1)
        for k in range(K):
            ax.plot(range(len(feats)), Wp[k], marker="o", color=cols[k], ls="-", ms=8, lw=3, label=f"State {k + 1}")
        title = "GLM weights (pooled dark, per-session faint)"
    else:
        mean, std = weights.mean(0), weights.std(0)
        for k in range(K):
            ax.errorbar(range(len(feats)), mean[k], yerr=std[k], fmt="o", color=cols[k], ms=5, lw=1.5, capsize=6, elinewidth=2)
            ax.plot(range(len(feats)), mean[k], marker="o", color=cols[k], ls="-", ms=8, lw=2.5, label=f"State {k + 1}")
        title = "GLM weights (mean +/- std, per-session faint)"

    ax.axhline(0, color="k", alpha=0.5, ls="--")
    ax.set_xticks(range(len(feats)))
    ax.set_xticklabels(feats, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("GLM weights", fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8, frameon=False)
    despine(ax)


def plot_psychometric(ax, bundle):
    """Actual vs model P(choice = 1) against the stimulus, pooled over the group's sessions."""
    stim_all, actual_all, model_all = [], [], []
    for sid, model in bundle["model"]["models"].items():
        inputs, choices, mask, valid, stimulus = session_design(bundle, sid)
        p1 = choice_prob(model, inputs, choices, mask)
        stim_all.append(stimulus[valid])
        actual_all.append(choices.ravel()[valid])
        model_all.append(p1[valid])
    stim_all = np.concatenate(stim_all)
    actual_all = np.concatenate(actual_all)
    model_all = np.concatenate(model_all)

    bins = np.unique(stim_all)
    a_mean = np.array([actual_all[stim_all == b].mean() for b in bins])
    a_sem = np.array([actual_all[stim_all == b].std() / np.sqrt(max((stim_all == b).sum(), 1)) for b in bins])
    m_mean = np.array([model_all[stim_all == b].mean() for b in bins])

    ax.errorbar(bins, a_mean, yerr=a_sem, fmt="o", color="k", capsize=3, ms=6, label="data")
    ax.plot(bins, m_mean, "-", color="crimson", lw=2.5, label="model")
    ax.axhline(0.5, color="grey", ls=":")
    ax.axvline(0, color="grey", ls=":")
    ax.set_xlabel("stimulus (signed coherence)", fontsize=12)
    ax.set_ylabel("P(choice = 1)", fontsize=12)
    ax.set_title("psychometric: data vs model", fontsize=12)
    ax.legend(fontsize=9, frameon=False)
    despine(ax)


def plot_transition(ax, bundle):
    """Across-session mean transition matrix."""
    K = bundle["best_k"]
    mats = np.array([np.array(m.transitions.transition_matrix) for m in bundle["model"]["models"].values()])
    mean_T = mats.mean(0)
    im = ax.imshow(mean_T, cmap="Blues", vmin=0, vmax=1)
    for i in range(K):
        for j in range(K):
            ax.text(j, i, f"{mean_T[i, j]:.2f}", ha="center", va="center", fontsize=9,
                    color="white" if mean_T[i, j] > 0.5 else "black")
    ax.set_xticks(range(K))
    ax.set_yticks(range(K))
    ax.set_xticklabels([f"S{k + 1}" for k in range(K)])
    ax.set_yticklabels([f"S{k + 1}" for k in range(K)])
    ax.set_xlabel("to state", fontsize=12)
    ax.set_ylabel("from state", fontsize=12)
    ax.set_title("mean transition matrix", fontsize=12)
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


def plot_posterior(ax, bundle):
    """Fractional state occupancy: per session (faint) and across-session mean (dark)."""
    K = bundle["best_k"]
    cols = state_colors(K)
    occ = []
    for sid, model in bundle["model"]["models"].items():
        inputs, choices, mask, valid, _ = session_design(bundle, sid)
        Ez = model.expected_states(choices, input=inputs, mask=mask)[0]
        occ.append(Ez[valid].mean(0))   # (K,) mean posterior over valid trials
        ax.plot(range(K), occ[-1], marker="o", color="grey", alpha=0.2, lw=1, ms=4)
    occ = np.array(occ)
    mean, std = occ.mean(0), occ.std(0)
    ax.errorbar(range(K), mean, yerr=std, fmt="-", color="k", lw=2, capsize=5, zorder=3)
    for k in range(K):
        ax.plot(k, mean[k], "o", color=cols[k], ms=11, zorder=4)
    ax.set_xticks(range(K))
    ax.set_xticklabels([f"S{k + 1}" for k in range(K)])
    ax.set_ylim(0, 1)
    ax.set_ylabel("mean posterior occupancy", fontsize=12)
    ax.set_title("state occupancy (per session + mean)", fontsize=12)
    despine(ax)

In [ ]:
# Restrict to specific model_name keys, or leave None to plot every fitted model.
MODELS_TO_PLOT = None
MODELS_TO_PLOT = ["standardized_stimulus__global_pooled_cv"]

for model_name, model_path in MODEL_PATHS.items():
    if MODELS_TO_PLOT is not None and model_name not in MODELS_TO_PLOT:
        continue
    finals = load_finals(model_path)
    if not finals:
        continue
    cv_type = "pooled_cv" if model_name.endswith("__global_pooled_cv") else "session_cv"

    for group in sorted(finals, key=lambda g: order_map.get(g, float("inf"))):
        bundle = finals[group]
        fig, axs = plt.subplots(2, 2, figsize=(16, 12))
        plot_weights(axs[0, 0], bundle)
        plot_psychometric(axs[0, 1], bundle)
        plot_transition(axs[1, 0], bundle)
        plot_posterior(axs[1, 1], bundle)
        fig.suptitle(f"{model_label(model_name)}  |  {group}   [{cv_type}, best_k={bundle['best_k']}]", fontsize=16, y=1.0)
        plt.tight_layout()
        plt.show()